# Predictive Campus Life Wellness Sentinel
## Random Forest - Model Training, Evaluation & Serialization

### Objective
This notebook loads the generated student behavioral dataset,
prepares the machine learning features, trains the Random Forest model,
evaluates its performance, compares it with XGBoost, and saves the
final trained Random Forest model for future inference.

### Main Activities
1. Load the dataset
2. Prepare features and target
3. Perform student-level train-test splitting
4. Train Random Forest
5. Evaluate the model
6. Compare with XGBoost
7. Analyze feature importance
8. Serialize the final Random Forest model

In [31]:
# ============================================================
# NOTEBOOK 2
# RandomForest_train
# Predictive Campus Life Wellness Sentinel
# ============================================================

# Import operating system utilities
import os

# Import Joblib for saving/loading the trained model
import joblib

# Import numerical and data manipulation libraries
import numpy as np
import pandas as pd

# Import Random Forest
from sklearn.ensemble import RandomForestClassifier

# Import train-test splitting
from sklearn.model_selection import train_test_split

# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [32]:
# ============================================================
# UPLOAD DATASET
# ============================================================

from google.colab import files

# Open the file upload dialog
uploaded = files.upload()

# Get the name of the uploaded file automatically
DATA_PATH = next(iter(uploaded))

print("Dataset uploaded successfully!")
print("File:", DATA_PATH)

Saving student_wellness_prediction_dataset.csv to student_wellness_prediction_dataset (2).csv
Dataset uploaded successfully!
File: student_wellness_prediction_dataset (2).csv


In [33]:
# ============================================================
# LOAD DATASET
# ============================================================

# Load uploaded CSV into Pandas DataFrame
df = pd.read_csv(DATA_PATH)

# Display dataset information
print("Dataset loaded successfully!")
print("Shape:", df.shape)

# Display first five rows
display(df.head())

Dataset loaded successfully!
Shape: (11000, 17)


,student_id,week,facility_usage,dining_activity,event_participation,club_participation,residence_engagement,recreation_activity,social_interactions,communication_activity,sleep_quality,academic_engagement,campus_engagement_score,social_isolation_score,engagement_change_pct,rolling_3_week_engagement,risk_label
0,10001,1,44.977101,44.777536,1,2,48.115146,33.427031,12.666752,19.446255,22.854321,27.737025,39.418087,70.198417,NaN,39.418087,1
1,10001,2,37.071197,45.129597,1,2,66.702819,43.718839,15.606497,14.901867,35.984585,54.460602,46.243754,66.534791,17.316081,42.830920,2
2,10001,3,44.355976,31.771601,1,2,34.047898,41.494406,5.197051,10.025365,43.955736,52.370869,34.240376,81.262044,-25.956755,39.967406,1
3,10001,4,47.916939,38.976086,1,2,22.422335,40.601190,17.906015,23.320972,50.930447,52.536182,43.258960,70.643535,26.339033,41.247697,1
4,10001,5,41.961635,26.905590,0,1,49.184727,48.738353,21.776601,17.562421,52.022452,48.602564,45.121073,68.472126,4.304572,40.873470,2


In [34]:
# ============================================================
# DATASET VALIDATION
# ============================================================

print("=" * 60)
print("DATASET VALIDATION")
print("=" * 60)

# Number of rows and columns
print("\nRows:", df.shape[0])
print("Columns:", df.shape[1])

# Number of unique students
print("Unique students:", df["student_id"].nunique())

# Number of weeks
print("Weeks:", df["week"].nunique())

# Check required target column
print("\nTarget column exists:", "risk_label" in df.columns)

# Display all columns
print("\nColumns:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

DATASET VALIDATION

Rows: 11000
Columns: 17
Unique students: 1000
Weeks: 11

Target column exists: True

Columns:
1. student_id
2. week
3. facility_usage
4. dining_activity
5. event_participation
6. club_participation
7. residence_engagement
8. recreation_activity
9. social_interactions
10. communication_activity
11. sleep_quality
12. academic_engagement
13. campus_engagement_score
14. social_isolation_score
15. engagement_change_pct
16. rolling_3_week_engagement
17. risk_label


In [35]:
# ============================================================
# DATASET INFORMATION
# ============================================================

print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

# Display missing values in each column
print(df.isnull().sum())

print("\n" + "=" * 60)
print("RISK DISTRIBUTION")
print("=" * 60)

# Convert numeric labels to readable names
risk_distribution = (
    df["risk_label"]
    .map({
        0: "Low",
        1: "Medium",
        2: "High"
    })
    .value_counts()
)

print(risk_distribution)

MISSING VALUES
student_id                      0
week                            0
facility_usage                  0
dining_activity                 0
event_participation             0
club_participation              0
residence_engagement            0
recreation_activity             0
social_interactions             0
communication_activity          0
sleep_quality                   0
academic_engagement             0
campus_engagement_score         0
social_isolation_score          0
engagement_change_pct        1000
rolling_3_week_engagement       0
risk_label                      0
dtype: int64

RISK DISTRIBUTION
risk_label
Medium    6750
High      4203
Low         47
Name: count, dtype: int64


In [36]:
# ============================================================
# FEATURE SELECTION
# ============================================================

FEATURE_COLUMNS = [
    "facility_usage",
    "dining_activity",
    "event_participation",
    "club_participation",
    "residence_engagement",
    "recreation_activity",
    "social_interactions",
    "communication_activity",
    "sleep_quality",
    "academic_engagement",
    "campus_engagement_score",
    "social_isolation_score",
    "engagement_change_pct",
    "rolling_3_week_engagement"
]

# Define target column
TARGET_COLUMN = "risk_label"

# Verify that all required columns exist
required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Create feature matrix
X = df[FEATURE_COLUMNS].copy()

# Create target vector
y = df[TARGET_COLUMN].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nNumber of features:", len(FEATURE_COLUMNS))

print("\nFeatures:")
for i, feature in enumerate(FEATURE_COLUMNS, start=1):
    print(f"{i}. {feature}")

Feature matrix shape: (11000, 14)
Target shape: (11000,)

Number of features: 14

Features:
1. facility_usage
2. dining_activity
3. event_participation
4. club_participation
5. residence_engagement
6. recreation_activity
7. social_interactions
8. communication_activity
9. sleep_quality
10. academic_engagement
11. campus_engagement_score
12. social_isolation_score
13. engagement_change_pct
14. rolling_3_week_engagement


In [37]:
# ============================================================
# TARGET LABEL VALIDATION
# ============================================================

print("=" * 60)
print("TARGET LABEL VALIDATION")
print("=" * 60)

print("\nUnique target values:")
print(sorted(y.unique()))

# Verify only expected classes exist
expected_classes = {0, 1, 2}

actual_classes = set(y.unique())

if not actual_classes.issubset(expected_classes):
    raise ValueError(
        f"Unexpected target classes found: {actual_classes}"
    )

print("\nRisk mapping:")
print("0 = Low")
print("1 = Medium")
print("2 = High")

print("\nTarget distribution:")
print(y.value_counts().sort_index())

TARGET LABEL VALIDATION

Unique target values:
[np.int64(0), np.int64(1), np.int64(2)]

Risk mapping:
0 = Low
1 = Medium
2 = High

Target distribution:
risk_label
0      47
1    6750
2    4203
Name: count, dtype: int64


In [38]:
# ============================================================
# STUDENT-LEVEL TRAIN TEST SPLIT
# ============================================================

# Get unique student IDs
unique_students = df["student_id"].unique()

# Split students into 80% training and 20% testing
train_students, test_students = train_test_split(
    unique_students,
    test_size=0.20,
    random_state=42
)

# Create training dataframe
train_df = df[
    df["student_id"].isin(train_students)
].copy()

# Create testing dataframe
test_df = df[
    df["student_id"].isin(test_students)
].copy()

# Create training features and target
X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET_COLUMN].copy()

# Create testing features and target
X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[TARGET_COLUMN].copy()

print("=" * 60)
print("STUDENT-LEVEL SPLIT")
print("=" * 60)

print("\nTotal students:", len(unique_students))
print("Training students:", len(train_students))
print("Testing students:", len(test_students))

print("\nTraining records:", len(train_df))
print("Testing records:", len(test_df))

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)

STUDENT-LEVEL SPLIT

Total students: 1000
Training students: 800
Testing students: 200

Training records: 8800
Testing records: 2200

X_train: (8800, 14)
X_test : (2200, 14)


In [39]:
# ============================================================
# HANDLE MISSING VALUES
# ============================================================

# Replace infinite values with NaN
X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)

# Calculate medians ONLY from training data
train_medians = X_train.median()

# Fill missing training values
X_train = X_train.fillna(train_medians)

# Fill missing testing values using training medians
X_test = X_test.fillna(train_medians)

print("Missing-value preprocessing completed.")

print("\nMissing values in X_train:")
print(X_train.isnull().sum().sum())

print("\nMissing values in X_test:")
print(X_test.isnull().sum().sum())

Missing-value preprocessing completed.

Missing values in X_train:
0

Missing values in X_test:
0


In [40]:
# ============================================================
# TRAIN / TEST CLASS DISTRIBUTION
# ============================================================

print("=" * 60)
print("TRAINING CLASS DISTRIBUTION")
print("=" * 60)

print(
    y_train
    .map({
        0: "Low",
        1: "Medium",
        2: "High"
    })
    .value_counts()
)

print("\n" + "=" * 60)
print("TESTING CLASS DISTRIBUTION")
print("=" * 60)

print(
    y_test
    .map({
        0: "Low",
        1: "Medium",
        2: "High"
    })
    .value_counts()
)

TRAINING CLASS DISTRIBUTION
risk_label
Medium    5414
High      3354
Low         32
Name: count, dtype: int64

TESTING CLASS DISTRIBUTION
risk_label
Medium    1336
High       849
Low         15
Name: count, dtype: int64


In [41]:
# ============================================================
# RANDOM FOREST MODEL
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=2,
    class_weight={
        0: 1.0,   # Low
        1: 1.0,   # Medium
        2: 4.0    # High
    },
    random_state=42,
    n_jobs=-1
)

# Train model
rf_model.fit(
    X_train,
    y_train
)

print("Random Forest training completed successfully!")

Random Forest training completed successfully!


In [42]:
# ============================================================
# RANDOM FOREST EVALUATION
# ============================================================

# Generate predictions
rf_pred = rf_model.predict(X_test)

# Generate prediction probabilities
rf_probability = rf_model.predict_proba(X_test)

# Calculate accuracy
rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

print("=" * 70)
print("RANDOM FOREST RESULTS")
print("=" * 70)

print(
    f"\nAccuracy: {rf_accuracy * 100:.2f}%"
)

# Classification report
print("\nClassification Report:")

print(
    classification_report(
        y_test,
        rf_pred,
        labels=[0, 1, 2],
        target_names=[
            "Low",
            "Medium",
            "High"
        ],
        zero_division=0
    )
)

# Confusion matrix
print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        rf_pred,
        labels=[0, 1, 2]
    )
)

RANDOM FOREST RESULTS

Accuracy: 60.27%

Classification Report:
              precision    recall  f1-score   support

         Low       0.00      0.00      0.00        15
      Medium       0.79      0.48      0.60      1336
        High       0.49      0.80      0.61       849

    accuracy                           0.60      2200
   macro avg       0.43      0.43      0.40      2200
weighted avg       0.67      0.60      0.60      2200


Confusion Matrix:
[[  0   4  11]
 [  0 646 690]
 [  0 169 680]]


In [43]:
# ============================================================
# RANDOM FOREST ROC-AUC
# ============================================================

rf_auc = roc_auc_score(
    y_test,
    rf_probability,
    multi_class="ovr",
    average="weighted"
)

print(
    f"Weighted ROC-AUC: {rf_auc:.4f}"
)

Weighted ROC-AUC: 0.7210


In [44]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({
    "Feature": FEATURE_COLUMNS,
    "Importance": rf_model.feature_importances_
})

# Sort by importance
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("Feature Importance:")

display(importance_df)

Feature Importance:


,Feature,Importance
0,engagement_change_pct,0.217349
1,campus_engagement_score,0.107921
2,rolling_3_week_engagement,0.075890
3,sleep_quality,0.073436
4,social_interactions,0.070566
5,recreation_activity,0.067107
6,social_isolation_score,0.062677
7,communication_activity,0.061649
8,residence_engagement,0.060971
9,academic_engagement,0.054639


In [45]:
# ============================================================
# INSTALL XGBOOST
# ============================================================

!pip install -q xgboost

print("XGBoost installation completed.")

XGBoost installation completed.


In [46]:
# ============================================================
# XGBOOST MODEL
# ============================================================

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

# Train XGBoost
xgb_model.fit(
    X_train,
    y_train
)

# Generate predictions
xgb_pred = xgb_model.predict(X_test)

# Generate probabilities
xgb_probability = xgb_model.predict_proba(X_test)

# Calculate accuracy
xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)

# Calculate ROC-AUC
xgb_auc = roc_auc_score(
    y_test,
    xgb_probability,
    multi_class="ovr",
    average="weighted"
)

print("=" * 70)
print("XGBOOST RESULTS")
print("=" * 70)

print(
    f"\nAccuracy: {xgb_accuracy * 100:.2f}%"
)

print(
    f"Weighted ROC-AUC: {xgb_auc:.4f}"
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        xgb_pred,
        labels=[0, 1, 2],
        target_names=[
            "Low",
            "Medium",
            "High"
        ],
        zero_division=0
    )
)

XGBOOST RESULTS

Accuracy: 68.73%
Weighted ROC-AUC: 0.7308

Classification Report:
              precision    recall  f1-score   support

         Low       0.00      0.00      0.00        15
      Medium       0.71      0.83      0.77      1336
        High       0.64      0.47      0.54       849

    accuracy                           0.69      2200
   macro avg       0.45      0.43      0.44      2200
weighted avg       0.68      0.69      0.67      2200



In [47]:
# ============================================================
# MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        rf_accuracy,
        xgb_accuracy
    ],
    "Weighted ROC-AUC": [
        rf_auc,
        xgb_auc
    ]
})

# Convert scores to percentages for accuracy
comparison_df["Accuracy"] = (
    comparison_df["Accuracy"] * 100
)

display(comparison_df)

,Model,Accuracy,Weighted ROC-AUC
0,Random Forest,60.272727,0.720990
1,XGBoost,68.727273,0.730843


In [48]:
# ============================================================
# SAVE TRAINED RANDOM FOREST MODEL
# ============================================================

MODEL_PATH = (
    "/content/wellness_randomforest_pipeline.pkl"
)

# Package model + feature information + risk mapping
model_package = {
    "model": rf_model,

    "features": FEATURE_COLUMNS,

    "risk_mapping": {
        0: "Low",
        1: "Medium",
        2: "High"
    }
}

# Save package using Joblib
joblib.dump(
    model_package,
    MODEL_PATH
)

print("Model saved successfully!")
print("Model path:", MODEL_PATH)
print(
    "File exists:",
    os.path.exists(MODEL_PATH)
)

Model saved successfully!
Model path: /content/wellness_randomforest_pipeline.pkl
File exists: True


In [49]:
# ============================================================
# VERIFY SAVED MODEL
# ============================================================

loaded_package = joblib.load(
    MODEL_PATH
)

print("Saved model loaded successfully!")

print("\nStored features:")

for i, feature in enumerate(
    loaded_package["features"],
    start=1
):
    print(f"{i}. {feature}")

print("\nRisk mapping:")
print(
    loaded_package["risk_mapping"]
)

Saved model loaded successfully!

Stored features:
1. facility_usage
2. dining_activity
3. event_participation
4. club_participation
5. residence_engagement
6. recreation_activity
7. social_interactions
8. communication_activity
9. sleep_quality
10. academic_engagement
11. campus_engagement_score
12. social_isolation_score
13. engagement_change_pct
14. rolling_3_week_engagement

Risk mapping:
{0: 'Low', 1: 'Medium', 2: 'High'}


In [50]:
# ============================================================
# TEST SERIALIZED MODEL
# ============================================================

loaded_model = loaded_package["model"]

# Predict using loaded model
test_prediction = loaded_model.predict(
    X_test.iloc[:5]
)

print("Predictions from saved model:")

for prediction in test_prediction:

    print(
        prediction,
        "->",
        loaded_package["risk_mapping"][prediction]
    )

Predictions from saved model:
2 -> High
1 -> Medium
1 -> Medium
2 -> High
2 -> High


In [51]:
# ============================================================
# DOWNLOAD MODEL TO LOCAL COMPUTER
# ============================================================

from google.colab import files

# Download trained model
files.download(
    "/content/wellness_randomforest_pipeline.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>